# Calculus Refresher for Machine Learning

**Type:** Hard prerequisite -- you should already know this material.  
**Estimated time:** 30-45 minutes  
**What this covers:** Single-variable derivatives, differentiation rules (product, quotient, chain), partial derivatives, and numerical differentiation in Python.  

**Who this is for:** Students who took a calculus course but need a quick refresher before diving into ML. If most of this feels unfamiliar (not just rusty), work through the resources in the References section first.

**Why this matters for ML:** Gradient-based optimization is the engine behind training neural networks. Every time you hear "backpropagation," you're hearing "chain rule applied systematically." You need solid intuition for what a derivative *means* and how to compute them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['figure.dpi'] = 100

---
## 1. Derivative as Slope / Rate of Change

The derivative of $f$ at point $a$ is the **instantaneous rate of change** -- the slope of the tangent line:

$$f'(a) = \lim_{h \to 0} \frac{f(a + h) - f(a)}{h}$$

**Intuition:**
- Positive derivative $\Rightarrow$ function is increasing
- Negative derivative $\Rightarrow$ function is decreasing  
- Zero derivative $\Rightarrow$ local extremum (or saddle point)

In ML, the derivative tells us: *if I nudge this parameter slightly, how does the loss change?*

### Example: Visualizing a tangent line

For $f(x) = x^2$, the derivative is $f'(x) = 2x$. At $x = 1$, the slope is $2$.

In [ ]:
def f(x):
    return x**2

def f_prime(x):
    return 2 * x

x = np.linspace(-2, 3, 200)
a = 1.0  # point of tangency
slope = f_prime(a)
tangent = f(a) + slope * (x - a)

plt.plot(x, f(x), 'b-', label=r'$f(x) = x^2$')
plt.plot(x, tangent, 'r--', label=f'Tangent at x={a} (slope={slope})')
plt.plot(a, f(a), 'ko', markersize=6)
plt.ylim(-1, 6)
plt.xlabel('x'); plt.ylabel('f(x)')
plt.legend(); plt.title('Derivative = Slope of the Tangent Line')
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 1.1

Plot $f(x) = x^3 - 3x + 1$ and its tangent line at $x = -1$ and $x = 2$.  
Compute the derivative analytically first, then plot both tangent lines on the same figure.

In [ ]:
# YOUR CODE HERE

---
## 2. Common Derivatives

You should know these by heart:

| $f(x)$ | $f'(x)$ |
|---------|----------|
| $x^n$ | $n x^{n-1}$ |
| $e^x$ | $e^x$ |
| $\ln(x)$ | $1/x$ |
| $\sin(x)$ | $\cos(x)$ |
| $\cos(x)$ | $-\sin(x)$ |
| $a^x$ | $a^x \ln(a)$ |

**ML context:** The exponential and logarithm appear everywhere -- softmax, cross-entropy loss, log-likelihood.

### Example: Verifying derivatives numerically

We can check analytic derivatives using finite differences: $f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$

In [ ]:
def numerical_derivative(f, x, h=1e-7):
    """Central difference approximation."""
    return (f(x + h) - f(x - h)) / (2 * h)

# Verify: d/dx[e^x] = e^x
x0 = 2.0
analytic = np.exp(x0)
numerical = numerical_derivative(np.exp, x0)
print(f"d/dx[e^x] at x={x0}:  analytic = {analytic:.8f},  numerical = {numerical:.8f}")
print(f"Absolute error: {abs(analytic - numerical):.2e}")

# Verify: d/dx[ln(x)] = 1/x
analytic_log = 1 / x0
numerical_log = numerical_derivative(np.log, x0)
print(f"\nd/dx[ln(x)] at x={x0}: analytic = {analytic_log:.8f},  numerical = {numerical_log:.8f}")
print(f"Absolute error: {abs(analytic_log - numerical_log):.2e}")

### Exercise 2.1

Verify the derivative of $\sin(x)$ at $x = \pi/4$ using the `numerical_derivative` function above. Compare against the analytic result $\cos(\pi/4)$.

In [ ]:
# YOUR CODE HERE

---
## 3. Product Rule and Quotient Rule

**Product rule:**
$$\frac{d}{dx}[f(x) \cdot g(x)] = f'(x) g(x) + f(x) g'(x)$$

**Quotient rule:**
$$\frac{d}{dx}\left[\frac{f(x)}{g(x)}\right] = \frac{f'(x) g(x) - f(x) g'(x)}{[g(x)]^2}$$

These come up less often in ML than the chain rule, but you need them for deriving gradient expressions by hand (e.g., softmax derivatives).

### Example: Product rule verification

$h(x) = x^2 \cdot \sin(x)$

$h'(x) = 2x \sin(x) + x^2 \cos(x)$

In [ ]:
h = lambda x: x**2 * np.sin(x)
h_prime = lambda x: 2*x * np.sin(x) + x**2 * np.cos(x)

x0 = 1.5
print(f"h'({x0}):  analytic = {h_prime(x0):.8f}")
print(f"h'({x0}):  numerical = {numerical_derivative(h, x0):.8f}")

x = np.linspace(0, 2 * np.pi, 200)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, h(x)); axes[0].set_title(r"$h(x) = x^2 \sin(x)$"); axes[0].grid(True, alpha=0.3)
axes[1].plot(x, h_prime(x), 'r'); axes[1].set_title(r"$h'(x) = 2x\sin(x) + x^2\cos(x)$"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

### Exercise 3.1

Compute the derivative of $q(x) = \dfrac{e^x}{x^2 + 1}$ using the quotient rule. Then verify numerically at $x = 0$ and $x = 1$.

In [ ]:
# YOUR CODE HERE

---
## 4. Chain Rule (Critical for Backpropagation)

If $y = f(g(x))$, then:

$$\frac{dy}{dx} = f'(g(x)) \cdot g'(x)$$

Or equivalently, with $u = g(x)$:

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx}$$

**This is the single most important rule for ML.** Backpropagation is just the chain rule applied repeatedly through a computational graph. When you chain multiple layers:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial a_n} \cdot \frac{\partial a_n}{\partial a_{n-1}} \cdots \frac{\partial a_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial w_1}$$

Each factor is a local derivative. The chain rule glues them together.

### Example: Chain rule step by step

Let $y = \sin(x^2)$. Here $f(u) = \sin(u)$ and $g(x) = x^2$.

$$\frac{dy}{dx} = \cos(x^2) \cdot 2x$$

In [ ]:
y = lambda x: np.sin(x**2)
y_prime = lambda x: np.cos(x**2) * 2 * x  # chain rule

x = np.linspace(-3, 3, 300)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, y(x), 'b-'); axes[0].set_title(r'$y = \sin(x^2)$'); axes[0].grid(True, alpha=0.3)
axes[1].plot(x, y_prime(x), 'r-'); axes[1].set_title(r"$y' = 2x\cos(x^2)$  (chain rule)"); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Numerical verification
for x0 in [0.5, 1.0, 2.0]:
    print(f"x={x0}: analytic={y_prime(x0):.6f}, numerical={numerical_derivative(y, x0):.6f}")

### Example: Nested chain rule (two compositions)

Let $y = e^{\sin(x^2)}$. We apply the chain rule twice:

$$\frac{dy}{dx} = e^{\sin(x^2)} \cdot \cos(x^2) \cdot 2x$$

In [ ]:
y2 = lambda x: np.exp(np.sin(x**2))
y2_prime = lambda x: np.exp(np.sin(x**2)) * np.cos(x**2) * 2 * x

for x0 in [0.5, 1.0, 1.5]:
    a = y2_prime(x0)
    n = numerical_derivative(y2, x0)
    print(f"x={x0}: analytic={a:.8f}, numerical={n:.8f}, error={abs(a-n):.2e}")

### Exercise 4.1

The **sigmoid function** is $\sigma(x) = \dfrac{1}{1 + e^{-x}}$.

1. Derive $\sigma'(x)$ using the chain rule. (Hint: write it as $(1 + e^{-x})^{-1}$ and apply the chain rule. You should get $\sigma'(x) = \sigma(x)(1 - \sigma(x))$.)
2. Implement `sigmoid` and `sigmoid_prime` in Python.
3. Verify numerically at $x = 0, 1, -2$.
4. Plot both $\sigma(x)$ and $\sigma'(x)$ on the interval $[-6, 6]$.

In [ ]:
# YOUR CODE HERE

### Exercise 4.2

Compute the derivative of $f(x) = \ln(1 + e^x)$ (this is the **softplus** function, commonly used in ML). Verify numerically. What familiar function does $f'(x)$ turn out to be?

In [ ]:
# YOUR CODE HERE

---
## 5. Partial Derivatives

For a function of multiple variables $f(x, y)$, the **partial derivative** with respect to $x$ treats $y$ as a constant:

$$\frac{\partial f}{\partial x} = \lim_{h \to 0} \frac{f(x + h, y) - f(x, y)}{h}$$

The **gradient** collects all partial derivatives into a vector:

$$\nabla f = \left(\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}\right)$$

**ML context:** A neural network loss $L(w_1, w_2, \ldots, w_n)$ depends on thousands/millions of parameters. Gradient descent updates each $w_i$ using $\frac{\partial L}{\partial w_i}$.

### Example: Partial derivatives of $f(x, y) = x^2 y + \sin(xy)$

$$\frac{\partial f}{\partial x} = 2xy + y\cos(xy), \qquad \frac{\partial f}{\partial y} = x^2 + x\cos(xy)$$

In [ ]:
def f2(x, y):
    return x**2 * y + np.sin(x * y)

def df_dx(x, y):
    return 2*x*y + y*np.cos(x*y)

def df_dy(x, y):
    return x**2 + x*np.cos(x*y)

# Numerical partial derivatives
def numerical_partial_x(f, x, y, h=1e-7):
    return (f(x + h, y) - f(x - h, y)) / (2 * h)

def numerical_partial_y(f, x, y, h=1e-7):
    return (f(x, y + h) - f(x, y - h)) / (2 * h)

x0, y0 = 1.0, 2.0
print(f"At ({x0}, {y0}):")
print(f"  df/dx: analytic = {df_dx(x0, y0):.8f}, numerical = {numerical_partial_x(f2, x0, y0):.8f}")
print(f"  df/dy: analytic = {df_dy(x0, y0):.8f}, numerical = {numerical_partial_y(f2, x0, y0):.8f}")

### Exercise 5.1

The **MSE loss** for a single prediction is $L(w, b) = (y - (wx + b))^2$, where $x$ and $y$ are fixed data points.

1. Compute $\frac{\partial L}{\partial w}$ and $\frac{\partial L}{\partial b}$ analytically.
2. For $x=2, y=5, w=1, b=1$: compute both partial derivatives analytically and numerically.

In [ ]:
# YOUR CODE HERE

---
## 6. Numerical Differentiation in Python

Three common finite difference formulas:

| Method | Formula | Error order |
|--------|---------|-------------|
| Forward | $\frac{f(x+h) - f(x)}{h}$ | $O(h)$ |
| Backward | $\frac{f(x) - f(x-h)}{h}$ | $O(h)$ |
| Central | $\frac{f(x+h) - f(x-h)}{2h}$ | $O(h^2)$ |

Central differences are more accurate for the same step size $h$. In practice, $h \approx 10^{-7}$ works well for `float64`.

### Example: Comparing accuracy of finite difference methods

In [ ]:
f = lambda x: np.sin(x)
f_exact = lambda x: np.cos(x)
x0 = 1.0
exact = f_exact(x0)

h_values = np.logspace(-1, -15, 30)
forward_errors = []
central_errors = []

for h in h_values:
    fwd = (f(x0 + h) - f(x0)) / h
    ctr = (f(x0 + h) - f(x0 - h)) / (2 * h)
    forward_errors.append(abs(fwd - exact))
    central_errors.append(abs(ctr - exact))

plt.loglog(h_values, forward_errors, 'b.-', label='Forward difference', alpha=0.7)
plt.loglog(h_values, central_errors, 'r.-', label='Central difference', alpha=0.7)
plt.xlabel('h (step size)'); plt.ylabel('Absolute error')
plt.title('Finite Difference Accuracy vs Step Size')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

print("Note: errors increase for very small h due to floating-point cancellation.")

### Exercise 6.1

Write a function `gradient_check(f, f_prime, x_values, h=1e-7)` that:
1. Takes a function `f`, its analytic derivative `f_prime`, and an array of test points.
2. Returns the maximum absolute error between the analytic and numerical (central difference) derivatives across all test points.

Test it with $f(x) = x^3$ and $f'(x) = 3x^2$ over `np.linspace(-2, 2, 10)`.

In [ ]:
# YOUR CODE HERE

---
## 7. Visualizing Derivatives: Function + Tangent Line

The tangent line to $f$ at $x = a$:

$$T(x) = f(a) + f'(a)(x - a)$$

This is a first-order Taylor approximation -- the best linear approximation near $a$.

### Example: Interactive-style tangent line plot

Plot $f(x) = e^{-x^2/2}$ (the Gaussian bell curve, unnormalized) with tangent lines at multiple points.

In [ ]:
f = lambda x: np.exp(-x**2 / 2)
f_prime = lambda x: -x * np.exp(-x**2 / 2)

x = np.linspace(-3, 3, 300)
tangent_points = [-1.5, -0.5, 0, 0.5, 1.5]
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(tangent_points)))

plt.figure(figsize=(10, 5))
plt.plot(x, f(x), 'k-', linewidth=2, label=r'$f(x) = e^{-x^2/2}$')

for a, c in zip(tangent_points, colors):
    slope = f_prime(a)
    tangent = f(a) + slope * (x - a)
    plt.plot(x, tangent, '--', color=c, alpha=0.8, label=f'Tangent at x={a} (slope={slope:.2f})')
    plt.plot(a, f(a), 'o', color=c, markersize=7)

plt.ylim(-0.5, 1.5); plt.xlim(-3, 3)
plt.xlabel('x'); plt.ylabel('f(x)')
plt.title('Tangent Lines at Various Points')
plt.legend(fontsize=8, loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()

### Exercise 7.1

Create a function `plot_with_tangent(f, f_prime, a, x_range=(-5, 5))` that:
1. Plots $f(x)$ over the given range.
2. Plots the tangent line at $x = a$.
3. Marks the point of tangency.
4. Displays the slope in the legend.

Test it with the **ReLU** function: $f(x) = \max(0, x)$. Try $a = -1$, $a = 0.5$, and $a = 2$. What happens at $a = 0$?

In [ ]:
# YOUR CODE HERE

---
## Self-Assessment

You should be able to answer all of these confidently. If you struggle with more than one, review the material before the course starts.

1. **What does $f'(a) = -3$ mean geometrically?** What does it tell you about $f$ near $x = a$?

2. **Compute by hand:** $\frac{d}{dx}\left[e^{3x^2}\right]$. Which rules did you use?

3. **Given** $f(x, y) = x^2 y + e^{xy}$, compute $\frac{\partial f}{\partial x}$ and $\frac{\partial f}{\partial y}$.

4. **Explain in one sentence** why the chain rule is essential for training neural networks.

If you could not answer these, work through the resources below before the course begins.

---
## References

1. **3Blue1Brown - Essence of Calculus** (video series): [https://www.youtube.com/playlist?list=PLZHQObOWTQDMsr9K-rj53DwVRMYO3t5Yr](https://www.youtube.com/playlist?list=PLZHQObOWTQDMsr9K-rj53DwVRMYO3t5Yr)  
   *Excellent visual intuition for derivatives, chain rule, and more. Start here if anything felt shaky.*

2. **Khan Academy - Differential Calculus**: [https://www.khanacademy.org/math/differential-calculus](https://www.khanacademy.org/math/differential-calculus)  
   *Structured practice with exercises and immediate feedback.*

3. **Mathematics for Machine Learning** (Deisenroth, Faisal, Ong), Chapter 5: [https://mml-book.github.io/](https://mml-book.github.io/)  
   *Free textbook. Chapter 5 covers vector calculus with ML applications.*